# Injecting Simulated Satellites to Quantify Dwarf Search Sensitivity in DP0

In [ ]:
import os
import sys
sys.path.append(os.path.expandvars('$HOME/software/simple_adl'))
sys.path.append(os.path.expandvars('$HOME/software/'))
import glob
import yaml
import time

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

import fitsio as fits

import matplotlib.pyplot as plt
import seaborn as sns

from astropy import units as u
from astropy.coordinates import SkyCoord

import simple_adl.survey
from simple_adl.search import search
from simple_adl.plot_osf import plot_osf
import simple_adl.load_data as load_data
from simple_adl.plot import plots
import dc2_satellite_census.code.selection_function as selection_function
from lsst.rsp import get_tap_service

from IPython.core.debugger import set_trace
import importlib

## Plotting

In [ ]:
# cmap = sns.cubehelix_palette(start=1.0, rot=-1.0, light=0.8, dark=0.2, hue=1.0, reverse=True, as_cmap=True)
cmap = sns.cubehelix_palette(start=3.0, rot=0.5, light=0.6, dark=0.2, hue=1.0, reverse=True, as_cmap=True)
# cmap = sns.cubehelix_palette(start=0.0, rot=0.0, light=0.7, dark=0.3, hue=1.0, reverse=True, as_cmap=True)
# cmap = sns.cubehelix_palette(start=0.5, rot=0.0, light=0.7, dark=0.3, hue=1.0, reverse=True, as_cmap=True)

cmap

In [ ]:
from lsst.rsp import get_tap_service

service = get_tap_service("tap")
assert service is not None
assert service.baseurl == "https://data.lsst.cloud/api/tap"

## Main loop over simulated satellites per position

In [ ]:
with open('config.yaml') as ymlfile:
    cfg = yaml.load(ymlfile, Loader=yaml.SafeLoader)
    survey = simple_adl.survey.Survey(cfg)

truth_match = False
if truth_match:
    sim_dir = '/project/shared/data/satsim/lsst_dc2_v7'
else:
    sim_dir = '/project/shared/data/satsim/lsst_dc2_v6'

population_file = glob.glob(os.path.join(sim_dir, '*population*'))    
for i, file in enumerate(population_file):
    
    sim_batch = load_data.get_sim_batch(pop_file=file)
    sim_population, sim_positions = load_data.sim_pop(file)
    for _, position in enumerate(sim_positions):
        try:
            real_data, sims_at_pos = load_data.load_field(sim_population, position,
                                                          service=service, truth_match=truth_match,
                                                          verbose=False)
        except TypeError:
            continue
        for mcid in sims_at_pos['MC_SOURCE_ID']:
            try:
                merged_data, sim_data = load_data.load_merge(mcid, real_data, 
                                                             position, survey=survey,
                                                             truth_match=truth_match,
                                                             verbose=False)
            except TypeError:
                continue
            print("----------DES----------")
            iso_selection = search(mcid, position, survey, merged_data, sims_at_pos, iso_survey='des',
                                   outfile='des_iso_pt2.csv', save=True, verbose=False)
            if iso_selection is None: continue
            plots(position, real_data, sim_data, merged_data, mcid, iso_selection, cmap)
            
            print("----------LSST----------")
            iso_selection = search(mcid, position, survey, merged_data, sims_at_pos, iso_survey='lsst',
                                   outfile='lsst_iso_pt2.csv', save=True, verbose=False)
            if iso_selection is None: continue
            plots(position, real_data, sim_data, merged_data, mcid, iso_selection, cmap)
            
            del merged_data

# Results plots

In [ ]:
#setting up sim dataframes

catalog_dir = '/project/shared/data/satsim/lsst_dc2_v7' 
results_dir = 'results_dir/v7_sims'
simsv7 = load_data.load_simdf(catalog_dir, results_dir)

catalog_dir = '/project/shared/data/satsim/lsst_dc2_v6' 
results_dir = 'results_dir/v6_sims'
simsv6 = load_data.load_simdf(catalog_dir, results_dir)

In [ ]:
# plot observational selection functions
plot_osf(simsv7, "Ideal star/galaxy separation", save=True, out_name='plots/ideal_osf.pdf')
plot_osf(simsv6, "Measured star/galaxy separation", save=True, out_name='plots/measured_osf.pdf')
#plot_osf(simsv6, "Measured star/galaxy separation (corrected)", save=False, out_name='measured_corr_osf.pdf', threshold=8.4)

# Analytic approximation for 50% detection efficiency contour

In [ ]:
def func(x,a,b,c):
    return a/(x - b) + c

In [ ]:
# Initial guess
P0=[[11.3 ,  10.0 ,  4.0  ],
    [22.6 ,  10.0 ,  4.3  ],
    [45.2 ,  6.0  ,  4.3  ],
    [90.5 ,  4.7  ,  4.3  ],
    [181.0,  2.1  ,  4.3  ],
    [362.0,  -0.7 ,  4.6  ],]

BOUNDS = [[0, -10, 3.5], #lower bounds for each column
          [512, 10, 5.0]] #upper bounds for each column

files = sorted(glob.glob('lsst_corrected_p50_d*.npy'))
results = []

for i,f in enumerate(files):
    dist = float(f.rsplit('_')[-1].rsplit('.',1)[0].strip('d'))
    data = np.load(f)
    sigma = 0.05*np.ones(len(data[:,1]))
    sigma[0] = 0.5
    sigma[-1] = 0.5
    #print sigma
    r = curve_fit(func,data[:,0],data[:,1],p0=P0[i],sigma=sigma,bounds=BOUNDS)
    results += [[dist]+r[0].tolist()]

results = np.asarray(results)
results = results[np.argsort(results[:,0])]

print('%-5s  %-5s  %-5s  %-5s'%('Dist','A0','Mv0','logr0'))
for r in results:
    print('[%-5.1f,  %-5.1f,  %-5.1f,  %-5.1f],'%tuple(r))

# XGBoost Classifier

In [ ]:
ssf_v6 = selection_function.SurveySelectionFunction('config.yaml', lsst_version=6)
ssf_v7 = selection_function.SurveySelectionFunction('config.yaml', lsst_version=7)

In [ ]:
ssf_v6.train_classifier()

In [ ]:
ssf_v7.train_classifier()

In [ ]:
ssf_v6.write_classifier('classifiers/v6_classifier2.0.model', force=True)
ssf_v7.write_classifier('classifiers/v7_classifier2.0.model', force=True)

In [ ]:
# example of classifier prediction

ssf.load_classifier('v6_classifier.model')                       # load classifier
size = 1                                                         # number of satellites
distance = np.random.uniform(8, 16, size)                        # kpc
abs_mag = np.random.uniform(-10, 2.5, size)                      # mag
r_physical = np.logspace(-3.5, 0, size)                          # kpc
x_eval = np.vstack([abs_mag, np.log10(r_physical), distance]).T
pdet = ssf.classifier.predict_proba(x_eval)[:,1]
print(pdet)                                                      # probability of detection

In [ ]:
pdet = ssf.classifier.predict_proba(ssf.X_train)[:,1]

In [ ]:
plt.hexbin(ssf.X_train[:,0], ssf.X_train[:,1], C=pdet, cmap='viridis')
plt.xlabel(r'$M_V$')
plt.ylabel(r'$log_{10}$(r_physical)')
plt.title('128 < D < 256 kpc')
plt.colorbar()
plt.show()

In [ ]:
ssf.load_classifier('v6_classifier.model')

### Bunch of plots for verification

In [ ]:
fig,axes = plt.subplots(2,3,figsize=(12,7))
plt.subplots_adjust(wspace=0, hspace=0)
axes[0,0].axes.get_xaxis().set_visible(False)
axes[0,1].axes.get_yaxis().set_visible(False)
axes[0,2].axes.get_yaxis().set_visible(False)
axes[1,2].axes.get_yaxis().set_visible(False)
axes[1,1].axes.get_yaxis().set_visible(False)
size = 20000
distance = np.random.uniform(8, 16, size)
abs_mag = np.random.uniform(-10, 2.5, size)
r_physical = np.logspace(0, 3.5, size)
x_eval = np.vstack([abs_mag, np.log10(r_physical), distance]).T
pdet = ssf.classifier.predict_proba(x_eval)[:,1]
axes[0,0].hexbin(abs_mag, np.log10(r_physical), C=pdet, cmap='viridis')
axes[0,0].set_xlabel(r'$M_V$')
axes[0,0].set_ylabel(r'$log_{10}$(r_physical)')
axes[0, 0].set_xlim(abs_mag.min(), abs_mag.max())
axes[0, 0].set_ylim(np.log10(r_physical.min()), np.log10(r_physical.max()))
plt.text(-37, 0.5, "8 < D < 16 kpc", fontsize=9)

distance = np.random.uniform(16, 32, size)
x_eval = np.vstack([abs_mag, np.log10(r_physical), distance]).T
pdet = ssf.classifier.predict_proba(x_eval)[:,1]
axes[0,1].hexbin(abs_mag, np.log10(r_physical), C=pdet, cmap='viridis')
axes[0,1].set_xlabel(r'$M_V$')
axes[0,1].set_ylabel(r'$log_{10}$(r_physical)')
axes[0, 1].set_xlim(abs_mag.min(), abs_mag.max())
axes[0, 1].set_ylim(np.log10(r_physical.min()), np.log10(r_physical.max()))
plt.text(-23, 0.5, "16 < D < 32 kpc", fontsize=9)

distance = np.random.uniform(32, 64, size)
x_eval = np.vstack([abs_mag, np.log10(r_physical), distance]).T
pdet = ssf.classifier.predict_proba(x_eval)[:,1]
axes[0,2].hexbin(abs_mag, np.log10(r_physical), C=pdet, cmap='viridis')
axes[0,2].set_xlabel(r'$M_V$')
axes[0,2].set_ylabel(r'$log_{10}$(r_physical)')
axes[0, 2].set_xlim(abs_mag.min(), abs_mag.max())
axes[0, 2].set_ylim(np.log10(r_physical.min()), np.log10(r_physical.max()))

plt.text(-10, 0.5, "32 < D < 64 kpc", fontsize=9)

distance = np.random.uniform(64, 128, size)
x_eval = np.vstack([abs_mag, np.log10(r_physical), distance]).T
pdet = ssf.classifier.predict_proba(x_eval)[:,1]
axes[1,0].hexbin(abs_mag, np.log10(r_physical), C=pdet, cmap='viridis')
axes[1,0].set_xlabel(r'$M_V$')
axes[1,0].set_ylabel(r'$log_{10}$(r_physical)')
axes[1, 0].set_xlim(abs_mag.min(), abs_mag.max())
axes[1, 0].set_ylim(np.log10(r_physical.min()), np.log10(r_physical.max()))

plt.text(-34, -3, "64 < D < 128 kpc", fontsize=9)

distance = np.random.uniform(128, 256, size)
x_eval = np.vstack([abs_mag, np.log10(r_physical), distance]).T
pdet = ssf.classifier.predict_proba(x_eval)[:,1]
axes[1,1].hexbin(abs_mag, np.log10(r_physical), C=pdet, cmap='viridis')
axes[1,1].set_xlabel(r'$M_V$')
axes[1,1].set_ylabel(r'$log_{10}$(r_physical)')
axes[1, 1].set_xlim(abs_mag.min(), abs_mag.max())
axes[1, 1].set_ylim(np.log10(r_physical.min()), np.log10(r_physical.max()))

plt.text(-20, -3, "128 < D < 256 kpc", fontsize=9)

distance = np.random.uniform(256, 512, size)
x_eval = np.vstack([abs_mag, np.log10(r_physical), distance]).T
pdet = ssf.classifier.predict_proba(x_eval)[:,1]
axes[1,2].hexbin(abs_mag, np.log10(r_physical), C=pdet, cmap='viridis')
axes[1,2].set_xlabel(r'$M_V$')
axes[1,2].set_ylabel(r'$log_{10}$(r_physical)')
axes[1, 2].set_xlim(abs_mag.min(), abs_mag.max())
axes[1, 2].set_ylim(np.log10(r_physical.min()), np.log10(r_physical.max()))

plt.text(-7, -3, "256 < D < 512 kpc", fontsize=9)

cax = fig.add_axes([0.93, 0.11, 0.02, 0.77])
cbar = fig.colorbar(axes[1, 2].hexbin(abs_mag, np.log10(r_physical), C=pdet, cmap='viridis'), cax=cax)
cbar.set_label('Probability (pdet)')

plt.show()

In [ ]:
def save_regions(path: str):
    """ Save queried DC2 regions to disc
    Parameters
    ----------
    path: str
        Location to save regions
    """
    service = get_tap_service()
    assert service is not None
    assert service.baseurl == "https://data.lsst.cloud/api/tap"
    sim_dir = '/project/shared/data/satsim/lsst_dc2_v6'
    population_file = glob.glob(os.path.join(sim_dir, '*population*'))
    with open('config.yaml') as ymlfile:
        cfg = yaml.load(ymlfile, Loader=yaml.SafeLoader)
        survey = simple_adl.survey.Survey(cfg)
        
    i = 0
    radius = 2
    for file in population_file:
        sim_population = fits.read(file)
        sim_positions = np.unique(sim_population[['RA', 'DEC']])
        for position in sim_positions:
            outfile = path + f'/region_{i}'
            if os.path.exists(outfile):
                i += 1
            else:
                print(f'Querying {position[0], position[1]}')
                real_data = query(service, position[0], position[1], radius)  # pd.DataFrame
                real_data.insert(loc=0, column='center_ra', value=position[0])
                real_data.insert(loc=1, column='center_dec', value=position[1])
                print(f'Saving region_{i}')
                real_data.to_csv(outfile)
                i += 1
            
    return